In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from matplotlib import pyplot as plt

# 1. PRZYGOTOWANIE DANYCH
df = pd.read_parquet('dataset.parquet')
model_name = 'model.pth'

In [14]:
# Wyodrębniamy unikalne zadania (tasks)
all_tasks = df['task'].unique().tolist()
num_tasks = len(all_tasks)

# Przekształcamy dane: jeden wiersz = jedna cząsteczka
# Tworzymy macierz etykiet (Y) z NaN tam, gdzie nie ma danych
df_pivot = df.pivot_table(index='smiles', columns='task', values='label')

# Dołączamy deskryptory (X) i informację o splicie
feature_cols = [c for c in df.columns if c not in ['label', 'task', 'split', 'smiles', 'mask_1', 'mask_2', 'mask_3', 'mask_4']]
df_features_split = df.groupby('smiles').agg({**{c: 'first' for c in feature_cols}, 'split': 'first'})

# Łączymy w jeden finalny zestaw
final_df = df_features_split.join(df_pivot)

# Normalizacja deskryptorów (Kluczowe przy różnych skalach dipole/energy)
scaler = StandardScaler()
final_df[feature_cols] = scaler.fit_transform(final_df[feature_cols])

# Podział na zbiory zgodnie z Twoją kolumną 'split'
train_df = final_df[final_df['split'] == 'train']
test_df = final_df[final_df['split'] == 'test']

# 2. PYTORCH DATASET
class ADMETDataset(Dataset):
    def __init__(self, dataframe, feature_list, task_list):
        self.X = torch.tensor(dataframe[feature_list].values, dtype=torch.float32)
        self.Y = torch.tensor(dataframe[task_list].values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# 3. ARCHITEKTURA MTL
class MultiTaskMLP(nn.Module):
    def __init__(self, input_size, num_tasks):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256)
        )
        # Każde zadanie dostaje własną warstwę wyjściową (Head)
        self.heads = nn.ModuleList([nn.Linear(256, 1) for _ in range(num_tasks)])

    def forward(self, x):
        features = self.backbone(x)
        # Zwracamy spakowane wyniki [batch, num_tasks]
        logits = torch.cat([head(features) for head in self.heads], dim=1)
        return logits

# 4. FUNKCJA STRATY Z MASKOWANIEM (IGNOROWANIE NaN)
def masked_bce_loss(logits, targets):
    # Maskujemy wartości NaN w targetach
    mask = ~torch.isnan(targets)
    # Liczymy stratę tylko dla istniejących etykiet
    criterion = nn.BCEWithLogitsLoss(reduction='none')
    loss = criterion(logits, torch.nan_to_num(targets)) # nan_to_num wrzuca 0, ale maska to wycina
    return (loss * mask.float()).sum() / mask.float().sum()

def generate_loss_plot(losses, filename='mtl_loss_plot.png'):
    plt.figure(figsize=(12, 6), dpi=100)
    plt.plot(losses, label='Total MTL Loss', color='royalblue')
    plt.title('Postęp Trenowania Modelu Multi-Task (ADMET)', fontsize=16, fontweight='bold')
    plt.xlabel('Iteracja / Epoka', fontsize=12)
    plt.ylabel('Wartość Funkcji Straty (Loss)', fontsize=12)
    plt.legend(fontsize=10) # Legenda
    plt.tight_layout()
    plt.savefig(filename)
    print(f"Wykres loss zapisany jako: {filename}")
    plt.close()

# 5. PĘTLA TRENINGOWA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używam urządzenia: {device}")

train_loader = DataLoader(ADMETDataset(train_df, feature_cols, all_tasks), batch_size=128, shuffle=True, drop_last=True, pin_memory=True)
test_loader = DataLoader(ADMETDataset(test_df, feature_cols, all_tasks), batch_size=128)

model = MultiTaskMLP(len(feature_cols), num_tasks).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

Używam urządzenia: cuda


In [19]:
train_losses = []
best_loss = 0
for epoch in range(100):
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = masked_bce_loss(outputs, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_losses.append(train_loss)
    generate_loss_plot(train_losses)
    if train_loss > best_loss:
        best_loss = train_loss
        torch.save(model.state_dict(), model_name)
    if len(train_losses) > 10 and min(train_losses[-11:-1]) > best_loss:
        break


    print(f"Epoch {epoch+1}/10 | Loss: {train_loss/len(train_loader):.4f}")

Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 1/10 | Loss: 0.1747
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 2/10 | Loss: 0.1706
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 3/10 | Loss: 0.1690
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 4/10 | Loss: 0.1637
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 5/10 | Loss: 0.1602
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 6/10 | Loss: 0.1616
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 7/10 | Loss: 0.1566
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 8/10 | Loss: 0.1486
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 9/10 | Loss: 0.1539
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 10/10 | Loss: 0.1473
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 11/10 | Loss: 0.1453
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 12/10 | Loss: 0.1451
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 13/10 | Loss: 0.1466
Wykres loss zapisany jako: mtl_loss_plot.png
Epoch 14/10 | Loss: 0.1442
W

In [25]:
from sklearn.metrics import roc_auc_score, accuracy_score
import numpy as np
import torch

def evaluate_per_task(model, data_loader, task_names, device):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            # Sigmoid zamienia logity na prawdopodobieństwa (0-1)
            outputs = torch.sigmoid(model(x))

            all_preds.append(outputs.cpu().numpy())
            all_targets.append(y.numpy())

    # Łączymy batche w jedną macierz
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    print("\n=== WYNIKI EWALUACJI PER TASK ===")
    print(f"{'Zadanie':25} | {'AUROC':8} | {'Accuracy':8}")
    print("-" * 45)

    results = {}

    for i, task_name in enumerate(task_names):
        y_true = all_targets[:, i]
        y_pred_proba = all_preds[:, i]

        # Filtrowanie NaN
        mask = ~np.isnan(y_true)
        y_true_clean = np.round(y_true[mask]).astype(int)
        y_pred_proba_clean = y_pred_proba[mask]

        # Dla Accuracy potrzebujemy twardych klas (0 lub 1)
        # Przyjmujemy próg decyzyjny 0.5
        y_pred_class_clean = (y_pred_proba_clean >= 0.5).astype(int)

        if len(np.unique(y_true_clean)) > 1:
            # Liczymy obie metryki
            auroc = roc_auc_score(y_true_clean, y_pred_proba_clean)
            acc = accuracy_score(y_true_clean, y_pred_class_clean)

            results[task_name] = {'auroc': auroc, 'accuracy': acc}
            print(f"{task_name:25} | {auroc:.4f}   | {acc:.4f}")
        else:
            print(f"{task_name:25} | Brak wystarczających danych")

    return results

model.load_state_dict(torch.load(model_name))
task_results = evaluate_per_task(model, test_loader, all_tasks, device)


=== WYNIKI EWALUACJI PER TASK ===
Zadanie                   | AUROC    | Accuracy
---------------------------------------------
bioavailability_ma        | 0.7523   | 0.7266
hia_hou                   | 0.9927   | 0.9558
pgp_broccatelli           | 0.9184   | 0.8472
bbb_martins               | 0.9060   | 0.8695
cyp2c9_veith              | 0.8828   | 0.8184
cyp2d6_veith              | 0.8554   | 0.8572
cyp3a4_veith              | 0.8763   | 0.7912
cyp2c9_substrate_carbonmangels | 0.6604   | 0.7429
cyp2d6_substrate_carbonmangels | 0.7403   | 0.7536
cyp3a4_substrate_carbonmangels | 0.6645   | 0.6232
hERG_Karim                | 0.8977   | 0.8198
ames                      | 0.8936   | 0.8202
dili                      | 0.9052   | 0.8241
